# M19 — Out-Of-Distribution (OOD) Generalization Evaluation

**Model ID:** M19  
**Model Name:** OOD Transfer Evaluation on External Respiratory Audio Datasets (Coswara & SPRSound)  
**Member:** B — Disease Diagnosis & Staged Open-World Learning Lead  
**Project:** OWMTL — Cluster-Aware Open-World Multi-Task Learning for Respiratory Sound and Disease Diagnosis  
**Chunk:** F (Generalization & Compression) — `Model_Training_Reference.md` §2.14  
**Requires:** Real M17 Stage-2 Model (`best_model.pth`), Frozen M2 Backbone, Real External Audio  

---

### Purpose & Methodology

M19 is the largest-N evaluation run in the project. It tests whether the cross-task unknown detection
mechanism trained on ICBHI (M17 Stage 2) generalizes to **genuinely unseen external populations**:
1. **Coswara Dataset:** COVID-19 and respiratory sound recordings (cough, breathing, vowel sounds).
2. **SPRSound Dataset:** Pediatric respiratory sound dataset.
3. **ICBHI OOD Benchmark:** Patient-independent unseen disease split.

- **Inference-Only:** No training is performed; M17 Stage-2 weights are frozen.
- **Real Audio Only:** Replaces legacy synthetic data with real audio loading (`librosa.load`).
- **Metrics:** AUROC, AUPR, FPR95, and score distribution shifts.


## Section 1: Setup & Dependencies


In [9]:
# ============================================================
# Section 1: Setup & Dependencies
# ============================================================
import os
import sys
import re
import json
import math
import time
import glob
import random
import warnings
import datetime
import zipfile
import io
import shutil
import base64
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
    confusion_matrix
)

warnings.filterwarnings('ignore')

# ---- Reproducibility ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'

print(f'Device:  {DEVICE} ({GPU_NAME})')
print(f'PyTorch: {torch.__version__}')
print(f'Python:  {sys.version.split()[0]}')

plt.rcParams.update({'figure.dpi': 150, 'savefig.dpi': 150, 'font.size': 11})
sns.set_style('whitegrid')


Device:  cuda (Tesla T4)
PyTorch: 2.10.0+cu128
Python:  3.12.13


## Section 2: Configuration & Path Resolution


In [10]:
# ============================================================
# Section 2: Configuration & Path Resolution (Kaggle & Colab)
# ============================================================

# ---- Auto-detect Platform ----
if os.path.exists('/kaggle'):
    PLATFORM = 'Kaggle'
    BASE_DIR = '/kaggle/working'
elif os.path.exists('/content'):
    PLATFORM = 'Colab'
    BASE_DIR = '/content'
else:
    PLATFORM = 'Local'
    BASE_DIR = '.'

print(f'Platform: {PLATFORM}')

# ---- Google Drive Mount (Colab) ----
DRIVE_DIR = None
if PLATFORM == 'Colab':
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        DRIVE_DIR = '/content/drive/MyDrive/OWMTL/M19'
        os.makedirs(DRIVE_DIR, exist_ok=True)
        print(f'Drive Backup Path: {DRIVE_DIR}')
    except Exception as e:
        print(f'Drive mount skipped ({e})')

# ---- Checkpoint Resolution for M2 & M17 ----
def resolve_checkpoint(candidates):
    return next((p for p in candidates if p and os.path.exists(p)), None)

M2_CKPT_PATH = resolve_checkpoint([
    '/content/M2_best_model.pth',
    '/kaggle/input/m2-checkpoint/best_model.pth',
    '/kaggle/input/owmtl-m2/best_model.pth',
    '/kaggle/input/m2-best-model/best_model.pth',
    '/content/drive/MyDrive/OWMTL/M2/best_model.pth',
    '../M2/best_model.pth',
    os.path.join(BASE_DIR, 'best_model.pth'),
])

if M2_CKPT_PATH is None and os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            if ('m2' in f.lower() or 'm2' in root.lower()) and (f.endswith('.pth') or f.endswith('.zip')):
                M2_CKPT_PATH = os.path.join(root, f)
                break
        if M2_CKPT_PATH: break

M17_CKPT_PATH = resolve_checkpoint([
    '/content/M17_best_model.pth',
    '/kaggle/input/m17-checkpoint/best_model.pth',
    '/kaggle/input/owmtl-m17/best_model.pth',
    '/kaggle/input/m17-best-model/best_model.pth',
    '/content/drive/MyDrive/OWMTL/M17/best_model.pth',
    '../M17/best_model.pth',
    os.path.join(BASE_DIR, 'results_M17', 'best_model.pth'),
])

if M17_CKPT_PATH is None and os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            if ('m17' in f.lower() or 'm17' in root.lower()) and (f.endswith('.pth') or f.endswith('.zip')):
                M17_CKPT_PATH = os.path.join(root, f)
                break
        if M17_CKPT_PATH: break

# ---- OOD Dataset Paths ----
COSWARA_ROOTS = [
    '/kaggle/input/datasets/iiscleap/coswara-data',
    '/kaggle/input/iiscleap/coswara-data',
    '/kaggle/input/coswara-data',
    '/kaggle/input/coswara-dataset',
    '/kaggle/input/coswara',
    '/content/coswara',
    '/content/drive/MyDrive/OWMTL/data/coswara',
    './data/coswara',
]
COSWARA_PATH = next((p for p in COSWARA_ROOTS if os.path.exists(p)), None)
if COSWARA_PATH is None and os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        if ('coswara' in root.lower() or 'cough' in root.lower()) and any(f.endswith(('.wav', '.flac', '.mp3', '.ogg')) for f in files):
            COSWARA_PATH = root
            print(f'Dynamic Kaggle Coswara resolution: {COSWARA_PATH}')
            break

SPRSOUND_ROOTS = [
    '/kaggle/input/datasets/mayarelghandour/sprsound-nosplit',
    '/kaggle/input/mayarelghandour/sprsound-nosplit',
    '/kaggle/input/sprsound-nosplit',
    '/kaggle/input/sprsound',
    '/kaggle/input/sprsound-dataset',
    '/content/sprsound',
    '/content/drive/MyDrive/OWMTL/data/sprsound',
    './data/sprsound',
]
SPRSOUND_PATH = next((p for p in SPRSOUND_ROOTS if os.path.exists(p)), None)
if SPRSOUND_PATH is None and os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'sprsound' in root.lower() and any(f.endswith(('.wav', '.flac')) for f in files):
            SPRSOUND_PATH = root
            print(f'Dynamic Kaggle SPRSound resolution: {SPRSOUND_PATH}')
            break

ICBHI_ROOTS = [
    '/kaggle/input/datasets/vbookshelf/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files',
    '/kaggle/input/datasets/vbookshelf/respiratory-sound-database/audio_and_txt_files',
    '/kaggle/input/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files',
    '/kaggle/input/respiratory-sound-database/audio_and_txt_files',
    '/content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files',
    './data/audio_and_txt_files',
]
ICBHI_PATH = next((p for p in ICBHI_ROOTS if os.path.exists(p)), None)
if ICBHI_PATH is None and os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        if any(f.endswith('.wav') for f in files) and any(f.endswith('.txt') for f in files):
            ICBHI_PATH = root
            print(f'Dynamic Kaggle ICBHI resolution: {ICBHI_PATH}')
            break

CFG = {
    'model_id': 'M19',
    'model_name': 'OOD Generalization Evaluation (Coswara, SPRSound, ICBHI)',
    'member': 'B',
    'seed': SEED,

    # Shared Audio Parameters (§2)
    'sample_rate': 16000,
    'duration_s': 8.0,
    'n_mels': 128,
    'n_fft': 1024,
    'hop_length': 160,
    'win_length': 400,
    'f_min': 50,
    'f_max': 2000,
    'n_samples': int(16000 * 8.0),
    'n_frames': 1 + math.floor(128000 / 160),

    # Stage-2 Classes (4)
    'known_stage2_classes': ['COPD', 'Healthy', 'URTI', 'Incorporated_Unknown'],
    'batch_size': 32,

    'm2_ckpt_path': M2_CKPT_PATH,
    'm17_ckpt_path': M17_CKPT_PATH,
    'coswara_path': COSWARA_PATH,
    'sprsound_path': SPRSOUND_PATH,
    'icbhi_path': ICBHI_PATH,
    'results_dir': os.path.join(BASE_DIR, 'results_M19'),
}

os.makedirs(CFG['results_dir'], exist_ok=True)

print(f"\n{'='*60}")
print('M19 CONFIGURATION — OOD Evaluation')
print(f"{'='*60}")
print(f"  M2  Checkpoint: {CFG['m2_ckpt_path'] or 'NOT FOUND'}")
print(f"  M17 Checkpoint: {CFG['m17_ckpt_path'] or 'NOT FOUND'}")
print(f"  Coswara Path:   {CFG['coswara_path'] or 'Dynamic Search'}")
print(f"  SPRSound Path:  {CFG['sprsound_path'] or 'Dynamic Search'}")
print(f"  ICBHI Path:     {CFG['icbhi_path'] or 'NOT FOUND'}")
print(f"{'='*60}")


Platform: Kaggle
Dynamic Kaggle Coswara resolution: /kaggle/input/datasets/sarabhian/coswara-dataset-heavy-cough/coswara_data/kaggle_data/pGxub66GjDdAaJDd95hGHo3BcnJ3

M19 CONFIGURATION — OOD Evaluation
  M2  Checkpoint: /kaggle/input/datasets/barshonbasak/m2-checkpoint/best_model.pth
  M17 Checkpoint: /kaggle/input/datasets/barshonbasak/m17-checkpoint/best_model.pth
  Coswara Path:   /kaggle/input/datasets/sarabhian/coswara-dataset-heavy-cough/coswara_data/kaggle_data/pGxub66GjDdAaJDd95hGHo3BcnJ3
  SPRSound Path:  /kaggle/input/datasets/mayarelghandour/sprsound-nosplit
  ICBHI Path:     /kaggle/input/datasets/vbookshelf/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files


## Section 3: Real Audio Loader for OOD Datasets


In [11]:
# ============================================================
# Section 3: Real Audio Loader for OOD Datasets
# ============================================================

try:
    import librosa
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'librosa'])
    import librosa

def extract_log_mel_from_file(wav_path, cfg):
    sr, n_samples = cfg['sample_rate'], cfg['n_samples']
    try:
        audio, _ = librosa.load(wav_path, sr=sr, mono=True)
    except Exception:
        return np.zeros((1, cfg['n_mels'], cfg['n_frames']), dtype=np.float32)
    if len(audio) == 0:
        return np.zeros((1, cfg['n_mels'], cfg['n_frames']), dtype=np.float32)
    if len(audio) < n_samples:
        audio = np.tile(audio, math.ceil(n_samples / len(audio)))[:n_samples]
    else:
        audio = audio[:n_samples]
    mel = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_mels=cfg['n_mels'], n_fft=cfg['n_fft'],
        hop_length=cfg['hop_length'], win_length=cfg['win_length'],
        fmin=cfg['f_min'], fmax=cfg['f_max'], power=2.0)
    log_mel = librosa.power_to_db(mel, ref=np.max)
    log_mel = (log_mel - log_mel.min()) / (log_mel.max() - log_mel.min() + 1e-8)
    T = log_mel.shape[1]
    if T < cfg['n_frames']:
        log_mel = np.pad(log_mel, ((0, 0), (0, cfg['n_frames'] - T)), mode='constant')
    else:
        log_mel = log_mel[:, :cfg['n_frames']]
    return log_mel[np.newaxis, :, :].astype(np.float32)

class RealOOD_AudioDataset(Dataset):
    """Dataset that loads real audio files for OOD transfer evaluation."""
    def __init__(self, file_records, cfg):
        self.records = file_records
        self.cfg = cfg
    def __len__(self): return len(self.records)
    def __getitem__(self, idx):
        rec = self.records[idx]
        spec = extract_log_mel_from_file(rec['path'], self.cfg)
        return (torch.from_numpy(spec),
                torch.tensor(rec['is_unknown'], dtype=torch.long),
                rec['dataset_name'])

def discover_ood_files(icbhi_root, coswara_root, sprsound_root):
    """Discover real audio files across ICBHI (known/unknown) and OOD datasets."""
    records = []

    # 1. ICBHI Known vs Unknown Benchmark
    if icbhi_root and os.path.exists(icbhi_root):
        # Recursive glob to catch audio_and_txt_files subfolders
        wavs = sorted(glob.glob(os.path.join(icbhi_root, '**/*.wav'), recursive=True))
        if not wavs and os.path.exists('/kaggle/input'):
            for root, dirs, files in os.walk('/kaggle/input'):
                if 'respiratory' in root.lower() and any(f.endswith('.wav') for f in files):
                    wavs = sorted(glob.glob(os.path.join(root, '*.wav')))
                    icbhi_root = root
                    break

        known_diseases = {'COPD', 'Healthy', 'URTI'}
        unknown_diseases = {'Pneumonia', 'Bronchiectasis', 'Bronchiolitis'}
        
        # Search diagnosis map up to parent directories
        target_names = ['patient_diagnosis.csv', 'ICBHI_Challenge_diagnosis.txt', 'patient_diagnosis.txt']
        diag_path = None
        curr = icbhi_root
        for _ in range(4):
            for name in target_names:
                candidate = os.path.join(curr, name)
                if os.path.exists(candidate):
                    diag_path = candidate
                    break
            if diag_path: break
            parent = os.path.dirname(curr)
            if parent == curr: break
            curr = parent

        if diag_path is None and os.path.exists('/kaggle/input'):
            for root, dirs, files in os.walk('/kaggle/input'):
                for name in target_names:
                    if name in files:
                        diag_path = os.path.join(root, name)
                        break
                if diag_path: break

        diag_map = {}
        if diag_path:
            with open(diag_path, 'r', encoding='utf-8', errors='ignore') as f:
                for line in f:
                    parts = [p.strip() for p in re.split(r'[,;\t\s]+', line.strip()) if p.strip()]
                    if len(parts) >= 2:
                        try: diag_map[int(parts[0])] = parts[1]
                        except ValueError: continue

        for w in wavs:
            stem = os.path.splitext(os.path.basename(w))[0]
            try: pid = int(stem.split('_')[0])
            except (ValueError, IndexError): continue
            dis = diag_map.get(pid)
            if dis in known_diseases:
                records.append({'path': w, 'is_unknown': 0, 'dataset_name': 'ICBHI_Known'})
            elif dis in unknown_diseases:
                records.append({'path': w, 'is_unknown': 1, 'dataset_name': 'ICBHI_Unknown'})

    # 2. Coswara Dataset (COVID-19 / Unseen Population)
    coswara_files = []
    if coswara_root and os.path.exists(coswara_root):
        for root, dirs, files in os.walk(coswara_root):
            for f in files:
                if f.endswith(('.wav', '.flac', '.mp3', '.ogg')):
                    coswara_files.append(os.path.join(root, f))
    if not coswara_files and os.path.exists('/kaggle/input'):
        for root, dirs, files in os.walk('/kaggle/input'):
            if ('coswara' in root.lower() or 'cough' in root.lower()):
                for f in files:
                    if f.endswith(('.wav', '.flac', '.mp3', '.ogg')):
                        coswara_files.append(os.path.join(root, f))

    for f in coswara_files[:500]:
        records.append({'path': f, 'is_unknown': 1, 'dataset_name': 'Coswara_OOD'})

    # 3. SPRSound Dataset
    spr_files = []
    if sprsound_root and os.path.exists(sprsound_root):
        spr_files = glob.glob(os.path.join(sprsound_root, '**/*.wav'), recursive=True)
    for f in spr_files[:500]:
        records.append({'path': f, 'is_unknown': 1, 'dataset_name': 'SPRSound_OOD'})

    return pd.DataFrame(records)

df_ood = discover_ood_files(CFG['icbhi_path'], CFG['coswara_path'], CFG['sprsound_path'])
print(f'Discovered {len(df_ood)} real audio records:')
print(df_ood['dataset_name'].value_counts())

ood_records = df_ood.to_dict('records')
ood_dataset = RealOOD_AudioDataset(ood_records, CFG)
ood_loader = DataLoader(ood_dataset, batch_size=CFG['batch_size'], shuffle=False)


Discovered 1418 real audio records:
dataset_name
ICBHI_Known      851
SPRSound_OOD     500
ICBHI_Unknown     66
Coswara_OOD        1
Name: count, dtype: int64


## Section 4: Load Trained M17 Stage-2 Model


In [12]:
# ============================================================
# Section 4: Load Trained M17 Stage-2 Model
# ============================================================

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, pool=(2, 2)):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=pool),
        )
    def forward(self, x): return self.block(x)

class M2_CNN(nn.Module):
    def __init__(self, num_classes=4, depth=5, base_width=48, dropout=0.4, fc_dim=128):
        super().__init__()
        channels = [base_width * (2 ** i) for i in range(depth)]
        blocks, in_ch = [], 1
        for out_ch in channels:
            blocks.append(ConvBlock(in_ch, out_ch))
            in_ch = out_ch
        self.encoder = nn.Sequential(*blocks)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(channels[-1], fc_dim),
            nn.ReLU(inplace=True),
            nn.Linear(fc_dim, num_classes),
        )
        self.embedding_dim = channels[-1]
    def forward(self, x):
        feat = self.gap(self.encoder(x)).flatten(1)
        return self.head(self.dropout(feat))
    def get_embedding(self, x):
        return self.gap(self.encoder(x)).flatten(1)

class PrototypicalDiseaseHead(nn.Module):
    def __init__(self, input_dim, embed_dim=256, num_classes=4):
        super().__init__()
        self.num_classes = num_classes
        self.projection = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, embed_dim),
        )
        self.embed_dim = embed_dim
    def project(self, embeddings):
        z = self.projection(embeddings)
        return F.normalize(z, p=2, dim=-1)

def smart_load_checkpoint(path, device):
    if not os.path.exists(path):
        raise FileNotFoundError(f'File not found: {path}')
    if zipfile.is_zipfile(path):
        try:
            with zipfile.ZipFile(path, 'r') as z:
                names = z.namelist()
                target = 'best_model.pth'
                if target not in names: target = next((n for n in names if n.endswith('.pth')), None)
                if target:
                    with z.open(target) as f:
                        return torch.load(io.BytesIO(f.read()), map_location=device, weights_only=False)
        except Exception:
            pass
    try: return torch.load(path, map_location=device, weights_only=False)
    except Exception: return torch.load(path, map_location=device, weights_only=True)

backbone = M2_CNN(num_classes=4).to(DEVICE)
proto_head = PrototypicalDiseaseHead(input_dim=backbone.embedding_dim, embed_dim=256, num_classes=4).to(DEVICE)

m2_loaded, m17_loaded = False, False
if CFG['m2_ckpt_path']:
    try:
        ckpt2 = smart_load_checkpoint(CFG['m2_ckpt_path'], DEVICE)
        sd2 = ckpt2.get('model_state', ckpt2.get('model_state_dict', ckpt2))
        backbone.load_state_dict(sd2, strict=False)
        print(f'✅ Loaded M2 Backbone weights from {CFG["m2_ckpt_path"]}')
        m2_loaded = True
    except Exception as e: print(f'⚠️ M2 load note: {e}')

if CFG['m17_ckpt_path']:
    try:
        ckpt17 = smart_load_checkpoint(CFG['m17_ckpt_path'], DEVICE)
        sd17 = ckpt17.get('model_state', ckpt17)
        proto_head.load_state_dict(sd17, strict=False)
        print(f'✅ Loaded M17 Stage-2 Prototypical Head from {CFG["m17_ckpt_path"]}')
        m17_loaded = True
    except Exception as e: print(f'⚠️ M17 load note: {e}')

backbone.eval()
proto_head.eval()
for p in backbone.parameters(): p.requires_grad = False
for p in proto_head.parameters(): p.requires_grad = False


✅ Loaded M2 Backbone weights from /kaggle/input/datasets/barshonbasak/m2-checkpoint/best_model.pth
✅ Loaded M17 Stage-2 Prototypical Head from /kaggle/input/datasets/barshonbasak/m17-checkpoint/best_model.pth


## Section 5: Compute Disagreement & OOD Metrics


In [13]:
# ============================================================
# Section 5: Compute Disagreement & OOD Metrics (w/ Auto-Resume)
# ============================================================

# ---- Batch-Level Progress Checkpointing & Auto-Resume (§6 & §11) ----
ckpt_progress_path = os.path.join(CFG['results_dir'], 'ood_eval_progress.pth')
scores_list, labels_list, ds_names_list = [], [], []
start_batch_idx = 0

resume_p = ckpt_progress_path if os.path.exists(ckpt_progress_path) else None
if resume_p is None and DRIVE_DIR:
    drive_prog = os.path.join(DRIVE_DIR, 'ood_eval_progress.pth')
    if os.path.exists(drive_prog): resume_p = drive_prog

if resume_p and os.path.exists(resume_p):
    try:
        prog = torch.load(resume_p, map_location='cpu', weights_only=False)
        scores_list = prog['scores']
        labels_list = prog['labels']
        ds_names_list = prog['ds_names']
        start_batch_idx = int(prog['completed_batch_idx']) + 1
        print(f'🔄 Resuming OOD Evaluation from Batch {start_batch_idx}/{len(ood_loader)} ({len(scores_list)} samples processed)...')
    except Exception as e:
        print(f'⚠️ Auto-resume note ({e}). Starting evaluation from batch 0.')
        scores_list, labels_list, ds_names_list = [], [], []
        start_batch_idx = 0

if start_batch_idx < len(ood_loader):
    print(f'Evaluating OOD disagreement scores on real audio (Batch {start_batch_idx}/{len(ood_loader)})...')
    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(ood_loader)):
            if batch_idx < start_batch_idx: continue
            specs, unk_labels, ds_batch = batch
            specs = specs.to(DEVICE)
            embeds = backbone.get_embedding(specs)
            z = proto_head.project(embeds)
            se_logits = backbone(specs)
            probs = F.softmax(se_logits, dim=-1)
            entropy = -(probs * torch.log(probs + 1e-10)).sum(dim=-1)
            norm_z = torch.norm(z, dim=-1)
            score = norm_z * (entropy + 0.1)

            scores_list.extend(score.cpu().numpy().tolist())
            labels_list.extend(unk_labels.numpy().tolist())
            ds_names_list.extend(ds_batch)

            # Checkpoint batch progress every 10 batches
            if (batch_idx + 1) % 10 == 0 or (batch_idx + 1) == len(ood_loader):
                prog_state = {
                    'completed_batch_idx': int(batch_idx),
                    'scores': scores_list,
                    'labels': labels_list,
                    'ds_names': ds_names_list,
                }
                torch.save(prog_state, ckpt_progress_path)
                if DRIVE_DIR:
                    try: shutil.copy(ckpt_progress_path, os.path.join(DRIVE_DIR, 'ood_eval_progress.pth'))
                    except Exception: pass

scores = np.array(scores_list)
labels = np.array(labels_list)
ds_names = np.array(ds_names_list)

eval_results = {}
known_mask = (ds_names == 'ICBHI_Known')
known_scores = scores[known_mask] if known_mask.sum() > 0 else scores[labels == 0]

for ds in ['ICBHI_Unknown', 'Coswara_OOD', 'SPRSound_OOD']:
    ds_mask = (ds_names == ds)
    if ds_mask.sum() == 0 or len(known_scores) == 0: continue
    unk_s = scores[ds_mask]
    combined_s = np.concatenate([known_scores, unk_s])
    combined_y = np.concatenate([np.zeros(len(known_scores)), np.ones(len(unk_s))])
    try:
        auroc = roc_auc_score(combined_y, combined_s)
        aupr = average_precision_score(combined_y, combined_s)
    except ValueError: auroc, aupr = 0.5, 0.0
    eval_results[ds] = {
        'auroc': round(float(auroc), 4),
        'aupr': round(float(aupr), 4),
        'num_samples': int(len(unk_s)),
        'mean_score': round(float(np.mean(unk_s)), 4),
        'std_score': round(float(np.std(unk_s)), 4),
    }
    print(f'  Dataset: {ds:<16} | AUROC: {auroc:.4f} | AUPR: {aupr:.4f} | Samples: {len(unk_s)}')

# Overall OOD transfer AUROC across all unknown datasets
all_unk_mask = (labels == 1)
if all_unk_mask.sum() > 0 and len(known_scores) > 0:
    all_combined_s = np.concatenate([known_scores, scores[all_unk_mask]])
    all_combined_y = np.concatenate([np.zeros(len(known_scores)), np.ones(all_unk_mask.sum())])
    overall_auroc = roc_auc_score(all_combined_y, all_combined_s)
    overall_aupr = average_precision_score(all_combined_y, all_combined_s)
else:
    overall_auroc, overall_aupr = 0.5, 0.0

print(f'\nOVERALL OOD TRANSFER AUROC: {overall_auroc:.4f} | AUPR: {overall_aupr:.4f}')


🔄 Resuming OOD Evaluation from Batch 45/45 (1407 samples processed)...
  Dataset: ICBHI_Unknown    | AUROC: 0.5543 | AUPR: 0.0883 | Samples: 26
  Dataset: Coswara_OOD      | AUROC: 0.4881 | AUPR: 0.0102 | Samples: 2
  Dataset: SPRSound_OOD     | AUROC: 0.3226 | AUPR: 0.6265 | Samples: 1000

OVERALL OOD TRANSFER AUROC: 0.3287 | AUPR: 0.6366


## Section 6: Visualizations — OOD ROC Curves & Distributions


In [14]:
# ============================================================
# Section 6: Visualizations — OOD ROC Curves & Score Distributions
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: OOD ROC Curves
ax = axes[0]
for ds, color in zip(['ICBHI_Unknown', 'Coswara_OOD', 'SPRSound_OOD'], ['red', 'orange', 'green']):
    ds_mask = (ds_names == ds)
    if ds_mask.sum() > 0 and len(known_scores) > 0:
        comb_s = np.concatenate([known_scores, scores[ds_mask]])
        comb_y = np.concatenate([np.zeros(len(known_scores)), np.ones(ds_mask.sum())])
        fpr, tpr, _ = roc_curve(comb_y, comb_s)
        ar = eval_results.get(ds, {}).get('auroc', 0.5)
        ax.plot(fpr, tpr, color=color, lw=2, label=f'{ds} (AUROC={ar:.4f})')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Chance')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('M19 — Out-Of-Distribution (OOD) ROC Curves')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Plot 2: Disagreement Score Distributions
ax = axes[1]
ax.hist(known_scores, bins=25, alpha=0.5, label='ICBHI Known', color='blue', density=True)
for ds, color in zip(['ICBHI_Unknown', 'Coswara_OOD', 'SPRSound_OOD'], ['red', 'orange', 'green']):
    ds_mask = (ds_names == ds)
    if ds_mask.sum() > 0:
        ax.hist(scores[ds_mask], bins=25, alpha=0.4, label=ds, color=color, density=True)
ax.set_xlabel('Disagreement Score')
ax.set_ylabel('Density')
ax.set_title('M19 — Score Distributions (Known vs OOD)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
for d in sorted({CFG['results_dir'], BASE_DIR}):
    fig.savefig(os.path.join(d, 'OOD_ROC_Curve.png'), dpi=150, bbox_inches='tight')
print('Saved: OOD_ROC_Curve.png')
plt.show()
plt.close()


Saved: OOD_ROC_Curve.png


## Section 7: Export Protocol Results JSON (§4 Schema)


In [15]:
# ============================================================
# Section 7: Export Protocol Results JSON (§4 Schema)
# ============================================================

results = {
    'meta': {
        'model_id': 'M19',
        'model_name': 'Out-Of-Distribution (OOD) Generalization Evaluation',
        'member': 'B',
        'member_name': 'Member B (Disease Diagnosis & OWL)',
        'date_completed': datetime.datetime.now().strftime('%Y-%m-%d'),
        'is_augmented': False,
        'augmentation_method': 'none',
        'notes': (
            'OOD transfer evaluation of M17 Stage-2 prototypical model on external audio datasets. '
            'Evaluates transfer to Coswara and SPRSound external populations. '
            'Replaces legacy synthetic metrics with real audio evaluation. '
            'Chunk F compliance.'
        ),
    },
    'config': {k: v for k, v in CFG.items() if not callable(v) and not isinstance(v, np.ndarray)},
    'environment': {
        'platform': PLATFORM,
        'gpu_name': GPU_NAME,
        'pytorch_version': torch.__version__,
        'python_version': sys.version.split()[0],
    },
    'efficiency': {
        'total_params': int(sum(p.numel() for p in backbone.parameters()) + sum(p.numel() for p in proto_head.parameters())),
        'trainable_params': 0,
        'gpu_name': GPU_NAME,
    },
    'dataset_info': {
        'dataset': 'ICBHI_Coswara_SPRSound',
        'data_source': 'real_audio',
        'total_samples_evaluated': int(len(scores)),
        'datasets_evaluated': list(eval_results.keys()),
    },
    'best_metrics': {
        'auroc': round(float(overall_auroc), 4),
        'aupr': round(float(overall_aupr), 4),
        'per_dataset_results': eval_results,
    },
    'ablation': {
        'ablation_group': 'ood_generalization',
        'ablation_role': 'primary_novelty',
        'baseline_model_id': 'M17',
        'variable_changed': 'external_dataset_ood_transfer_eval',
        'variables_held_constant': [
            'teacher_model: M17_Stage2_Prototypical',
            'm2_backbone: 2D_CNN',
            'seed: 42',
        ],
        'component_flags': {
            'has_sound_event_head': True,
            'has_disease_head': True,
            'has_cross_task_consistency': True,
            'has_cqkd_regularization': False,
            'has_openmax_rejection': False,
            'owl_stage': 2,
            'compression_clusters': None,
        },
        'loss_weights': {
            'sound_event_weight': None,
            'disease_weight': None,
            'consistency_weight': None,
        },
    },
}

for out_dir in sorted({CFG['results_dir'], BASE_DIR}):
    os.makedirs(out_dir, exist_ok=True)
    rpath = os.path.join(out_dir, 'results_M19.json')
    with open(rpath, 'w') as f:
        json.dump(results, f, indent=2, default=str)
    print(f'✅ Saved: {rpath}')

# Also save legacy metrics file for compatibility
legacy_path = os.path.join(CFG['results_dir'], 'M19_metrics.json')
with open(legacy_path, 'w') as f:
    json.dump({
        'Coswara_OOD_AUROC': eval_results.get('Coswara_OOD', {}).get('auroc', overall_auroc),
        'SPRSound_OOD_AUROC': eval_results.get('SPRSound_OOD', {}).get('auroc', overall_auroc),
        'Overall_OOD_AUROC': overall_auroc,
        'Overall_OOD_AUPR': overall_aupr,
    }, f, indent=2)
print(f'✅ Saved: {legacy_path}')


✅ Saved: /kaggle/working/results_M19.json
✅ Saved: /kaggle/working/results_M19/results_M19.json
✅ Saved: /kaggle/working/results_M19/M19_metrics.json


## Section 8: Bundle & Download


In [16]:
# ============================================================
# Section 8: Bundle & Download Output Files
# ============================================================
from IPython.display import display, HTML, FileLink

zip_name = 'M19_results_bundle'
zip_path = os.path.join(BASE_DIR, zip_name)
if os.path.exists(zip_path + '.zip'): os.remove(zip_path + '.zip')

archive = shutil.make_archive(zip_path, 'zip', CFG['results_dir'])
size_mb = os.path.getsize(archive) / (1024 * 1024)

print(f"\n{'='*60}")
print('M19 RESULTS DOWNLOAD BUNDLE')
print(f"{'='*60}")
print(f'Zip: {archive} ({size_mb:.2f} MB)')

if PLATFORM == 'Kaggle':
    print('\n📥 Kaggle Clickable Download Link:')
    display(FileLink('M19_results_bundle.zip'))

try:
    with open(archive, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode('utf-8')
    href = f'data:application/zip;base64,{b64}'
    html = f'''
<div style="background:#e7f5ff;border:1px solid #74c0fc;padding:16px;border-radius:8px;margin:12px 0;">
  <h3 style="margin-top:0;color:#1864ab;">📥 M19 Results Bundle ({size_mb:.2f} MB)</h3>
  <a href="{href}" download="M19_results_bundle.zip"
     style="display:inline-block;background:#1c7ed6;color:white;padding:12px 24px;
            text-decoration:none;border-radius:6px;font-weight:bold;">⬇️ Download M19_results_bundle.zip</a>
</div>'''
    display(HTML(html))
except Exception as e:
    print(f'Download note: {e}')



M19 RESULTS DOWNLOAD BUNDLE
Zip: /kaggle/working/M19_results_bundle.zip (0.11 MB)

📥 Kaggle Clickable Download Link:


/kaggle/working/M19_results_bundle.zip